# Silver Layer — Customer Processing

## Objective
Transform raw customer data into a clean, deduplicated Delta table for downstream analytics.

## Implementation
- Ingest customer changes incrementally from Bronze using Delta Change Data Feed (CDF).
- Apply data cleaning and deduplication.
- Use Delta MERGE to insert new customers and update existing records.
- Maintain customer data for downstream Gold dimensional modeling.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

bronze_table = "banking_project.bronze.bronze_customers"
silver_table = "banking_project.silver.silver_customers"

checkpoint_path = (
    "/Volumes/banking_project/silver/"
    "silver_pipeline_state/customers_cdf_checkpoint"
)

def merge_customers(batch_df, batch_id):

    if batch_df.isEmpty():
        return

    # Only inserts and final updated records
    df = batch_df.filter(
        F.col("_change_type").isin("insert", "update_postimage")
    )

    # Clean customer data
    df = (
        df.select(
            F.trim("customer_id").alias("customer_id"),
            F.trim("customer_name").alias("customer_name"),
            F.trim("customer_city").alias("customer_city"),
            "ingestion_timestamp",
            "source_file",
            "_commit_version"
        )
        .filter(
            F.col("customer_id").isNotNull() &
            (F.col("customer_id") != "")
        )
    )

    # Keep the latest change per customer
    window = Window.partitionBy("customer_id").orderBy(
        F.col("_commit_version").desc(),
        F.col("ingestion_timestamp").desc_nulls_last()
    )

    df = (
        df.withColumn("rn", F.row_number().over(window))
        .filter(F.col("rn") == 1)
        .drop("rn", "_commit_version")
    )

    # Upsert into existing Silver table
    (
        DeltaTable.forName(spark, silver_table)
        .alias("target")
        .merge(
            df.alias("source"),
            "target.customer_id = source.customer_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    spark.readStream
    .option("readChangeFeed", "true")
    .table(bronze_table)
    .writeStream
    .foreachBatch(merge_customers)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("Silver Customers CDF processing completed")